In [1]:
import sys
import os

# 设置你的 main.py 所在目录路径，例如：
project_dir = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/"

# 加入到 sys.path（如果尚未添加）
if project_dir not in sys.path:
    sys.path.append(project_dir)

# 检查是否添加成功
print("Updated sys.path:", sys.path)

Updated sys.path: ['/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/scripts', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python39.zip', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python3.9', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python3.9/lib-dynload', '', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python3.9/site-packages', '/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/']


In [2]:
import os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from lib.prune import (
    prune_wanda,
    check_sparsity,
    get_mask,
    prune_wandg_set_difference,
)
from lib.model_wrapper import prune_wanda_v2, prune_wandg
from lib.eval import eval_ppl, eval_zero_shot, eval_attack, load_prompt, load_dataset, format_prompt, _safe_generate, extract_answer
    
from vllm import LLM
import argparse
from tqdm import tqdm
from vllm import SamplingParams
from pathlib import Path
from typing import Dict, List
import json
import random

In [26]:
# 📌 设置参数（你原本命令行中给出的内容）
# 构造参数
args = argparse.Namespace(
    model="llama2-7b-chat-hf",
    model_base="llama2-7b-hf",
    seed=0,
    nsamples=2,
    sparsity_ratio=0.5,
    sparsity_type="unstructured",
    prune_method="prune_wandg_set_difference",  # 举例使用 wanda
    prune_data="GSM8K_cot0shot_120",  # 例如使用 GSM8K 数据
    use_diff=False,
    neg_prune=False,
    recover_from_base=False,
    p=0.5,
    q=0.5,
    top_k_heads=10,
    cache_dir="llm_weights",
    use_variant=False,
    save="temp_with_template_1",
    save_model=None,
    save_mask=None,
    dump_wanda_score=False,
    eval_zero_shot=True,
    eval_attack=True,
    save_attack_res=True,
    prune_part=False,
    disentangle=True,  # 注意：原 argparse 中是 --entangle_prompt_feat -> dest="disentangle", action="store_false"
    decouple_align_utility=False,
    decouple_align_misalign=False,
    rank=10,
    niter=20,
    prompt_method="cot0shot",
    dataset="GSM8K",
    role="math teacher",
    batch_size=64,
    eval_type="fixed",  # 新增参数
)


suffix = "weightonly"
cache_dir = "llm_weights"
save_dir = f"out/{args.model}/{args.sparsity_type}/{args.prune_method}_{suffix}/{args.prune_data}_with_template_1"
os.makedirs(save_dir, exist_ok=True)
args.save = save_dir

In [27]:
# 💾 使用 HuggingFace 下载模型到本地文件夹，并用 vllm 加载

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

from transformers import AutoModelForCausalLM, AutoTokenizer

# 定义本地保存路径
local_model_dir = "./llama2-7b-chat-hf-local"

# 如果本地没有模型，则下载
if not os.path.exists(local_model_dir):
    os.makedirs(local_model_dir, exist_ok=True)
    # 下载模型和分词器到本地
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-2-7b-chat-hf",
        torch_dtype="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-2-7b-chat-hf",
    )
    # 保存到本地
    model.save_pretrained(local_model_dir)
    tokenizer.save_pretrained(local_model_dir)

# vllm 支持直接加载本地文件夹
modeltype2path = {
    "llama2-7b-chat-hf": local_model_dir,
    "llama2-7b-hf": "meta-llama/Llama-2-7b-hf",  # 如有需要可同样下载
}

from vllm import LLM

vllm_model = LLM(
    model=local_model_dir,
    dtype="float16",
    swap_space=16,
)

INFO 08-16 14:32:38 llm_engine.py:72] Initializing an LLM engine with config: model='./llama2-7b-chat-hf-local', tokenizer='./llama2-7b-chat-hf-local', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, quantization=None, seed=0)
INFO 08-16 14:32:38 tokenizer.py:31] For some LLaMA V1 models, initializing the fast tokenizer may take a long time. To reduce the initialization time, consider using 'hf-internal-testing/llama-tokenizer' instead of the original tokenizer.


AssertionError: tensor model parallel group is already initialized

In [ ]:
import time
def apply_prompt_template(model_name, prompts):
    print(f"Applying prompt template for model: {model_name}")
    """
    根据不同的 model_name 应用聊天模板。
    支持单个字符串或字符串列表。
    """
    if isinstance(prompts, str):
        prompts = [prompts]

    formatted_prompts = []
    for prompt in prompts:
        if "llama2" in model_name.lower():
            # LLaMA 2 Chat 模版
            B_INST, E_INST = "[INST]", "[/INST]"
            B_SYS, E_SYS = "<<SYS>>\n", "\n<</SYS>>\n\n"
            system_prompt = "You are a helpful assistant."
            formatted = f"{B_INST} {B_SYS}{system_prompt}{E_SYS}{prompt} {E_INST}"
        elif "mistral" in model_name.lower():
            # Mistral 模版（以 <s> 和 [INST] 开头）
            formatted = f"<s>[INST] {prompt} [/INST]"
        elif "chatglm" in model_name.lower():
            # ChatGLM 模版
            formatted = f"[Round 1]\n问：{prompt}\n答："
        else:
            # 默认不加模板
            formatted = prompt

        formatted_prompts.append(formatted)

    return formatted_prompts


def _safe_generate(args, model, prompts, sampling_params, max_retry=5, backoff=10, add_template=False):
    """
    统一把 vLLM 的输出转成 List[str]。
    prompts 既可以是 str，也可以是 List[str]，最终都以 List[str] 返回。
    """
    if isinstance(prompts, str):
        prompts = [prompts]
    if add_template:
        prompts = apply_prompt_template(args.model, prompts)
    retry = 0
    while True:
        try:
            # vllm.LLM.generate 接收 List[str]
            raw = model.generate(prompts, sampling_params)    # type: List[RequestOutput]

            # 取每个 RequestOutput 的首个 candidate 文本
            clean = [
                (r.outputs[0].text if getattr(r, "outputs", None) else str(r)).strip()
                for r in raw
            ]
            return clean                                           # List[str]
        except Exception as e:
            retry += 1
            if retry > max_retry:
                raise RuntimeError(f"Generation failed after {max_retry} retries") from e
            wait = backoff * (2 ** (retry - 1))
            print(f"[WARN] Generate error: {e!s} | Retry {retry}/{max_retry} after {wait}s")
            time.sleep(wait)


In [ ]:
def eval_gsm8k_held_out(
    args,
    vllm_model,
    tokenizer=None,
    prune_data: str = "GSM8K_direct_120",
):
    prompt_tags = [p.strip() for p in args.prompt_method.split(",") if p.strip()]
    full_prompts = {
        tag: load_prompt(args.dataset, tag, do_role=args.role) for tag in prompt_tags
    }

    random.seed(args.seed)
    np.random.seed(args.seed)
    # 2) 提取所有 id（去掉可能为空的）
    data_file = f"/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K/heldout_500.jsonl"
    with open(data_file, "r") as fin:
        samples = [json.loads(line) for line in fin if "id" in json.loads(line)]
    ids = [s["id"] for s in samples if "id" in s and s["id"]]

    # 3) 传给 load_dataset
    data = load_dataset(
        args.dataset,
        args.nsamples,
        select_method="held_out",
        ids=ids,          # 这里就是 samples 中所有 id 的列表
    )

    # -------- ❷  设置 SamplingParams --------
    sampling_params = SamplingParams(
        temperature=0.0,
        top_p=1.0,
        max_tokens=1024,
        n=1,              # GSM8K 评测通常一个样本即可
        stop=None,        # 统一在 extract_answer 里截断
    )

    # -------- ❸  主循环：每种 prompt 独立评估 --------
    acc_dict: Dict[str, List[bool]] = {t: [] for t in prompt_tags}

    for tag in prompt_tags:
        if args.neg_prune:
            print("Negative pruning")
            outfile = (Path(args.save)
                    / f"gsm8k_top_{args.sparsity_ratio:.6f}_{args.prompt_method}_{args.eval_type}_prompt_{tag}.jsonl"
                    )
        else:
            print("Positive pruning")
            outfile = (Path(args.save)
                / f"gsm8k_bottom_{args.sparsity_ratio:.6f}_{args.prompt_method}_{args.eval_type}_prompt_{tag}.jsonl"
            )

        already_done = 0
        out_fh = open(outfile, "a")

        if outfile.exists():
            # JSONL 易于续写；记录已完成行数
            already_done = sum(1 for _ in open(outfile))
            if already_done >= len(data):
                print(f"[SKIP] {outfile.name} 已完成 ({already_done}/{len(data)})")
                out_fh.close()
                acc_dict[tag] = [
                    json.loads(line)["correct"] for line in open(outfile)
                ]
                continue
            print(f"[RESUME] {outfile.name}: 已有 {already_done} 条，继续评估 …")

        dataset_iter = data[already_done:]
        dataset_chunks = [
            dataset_iter[i : i + args.batch_size]
            for i in range(0, len(dataset_iter), args.batch_size)
        ]

        for chunk in tqdm(dataset_chunks, desc=f"Eval {tag}", ncols=80):
            # 组装输入
            messages = [format_prompt(full_prompts[tag], sample) for sample in chunk]
            outputs = _safe_generate(args, vllm_model, messages, sampling_params, add_template=True)

            preds = [
                extract_answer(out_text, sample, args.dataset)
                for out_text, sample in zip(outputs, chunk)
            ]

            for sample, out_text, pred in zip(chunk, outputs, preds):
                gold = sample["answer"]
                correct = pred == gold
                acc_dict[tag].append(correct)

                record = {
                    **sample,                       # 题目 & gold answer
                    "prompt_tag": tag,
                    "input": format_prompt(full_prompts[tag], sample),
                    "output": out_text,
                    "pred": pred,
                    "gold": gold,
                    "correct": correct,
                }
                out_fh.write(json.dumps(record, ensure_ascii=False) + "\n")
                out_fh.flush()

        out_fh.close()

    # -------- ❹  汇总指标 --------
    acc_summary = {
        tag: float(np.mean(acc)) if acc else 0.0 for tag, acc in acc_dict.items()
    }
    for tag, acc in acc_summary.items():
        n = len(acc_dict[tag])
        print(f"[ACC] {tag:15s}: {acc:.3%} ({int(acc*n)}/{n})")

    return acc_summary

In [32]:
save_filepath = os.path.join(args.save, f"log_{args.prune_method}.txt")

print(f"Evaluating GSM8K {args.prune_data} with {args.model}")
score = eval_gsm8k_held_out(
    args,
    vllm_model,
    None,
    prune_data=args.prune_data,
)



Evaluating GSM8K GSM8K_cot0shot_120 with llama2-7b-chat-hf
Positive pruning
[RESUME] gsm8k_bottom_0.500000_cot0shot_fixed_prompt_cot0shot.jsonl: 已有 0 条，继续评估 …


Eval cot0shot:   0%|                                      | 0/8 [00:00<?, ?it/s]


TypeError: _safe_generate() missing 1 required positional argument: 'sampling_params'